# Vectorless RAG (Reasoning-Based Tree Retrieval)

![Vectorless RAG Architecture](./images/Vectorless%20Rag.jpg)

---

## Core Concept

Unlike traditional RAG—which splits documents into fixed chunks, converts them into vector embeddings, and searches using cosine similarity—**Vectorless RAG** (e.g., *PageIndex*) preserves the document's natural structure.

It builds a hierarchical **JSON Tree Index** (similar to a smart Table of Contents) and uses an **LLM to reason and traverse** the tree step by step to find relevant information with page-level accuracy.

---

## Pipeline 1: Data Indexing (Offline / Building)

### 1. Document Ingestion

- Takes long, structured or unstructured documents, such as PDFs, textbooks, technical manuals, and research papers.
- Preserves the document's logical structure instead of blindly splitting paragraphs into fixed-size chunks.
- Identifies the document content and its page boundaries.

### 2. LLM Tree Builder

- An LLM analyzes the document's layout, headings, and subheadings to extract a logical hierarchy of sections and subsections.
- Generates concise summaries for each node so the tree captures what every section is about.
- Organizes the document into parent and child nodes.

### 3. JSON Tree Index (No Vector DB Needed)

- Stores the document hierarchy, node summaries, and exact page references (`p1`, `p2`, `p3`, etc.) in a lightweight JSON structure.
- Each parent node contains its relevant child nodes.
- The index allows the LLM to navigate the document without reading the entire document during every query.

**Example JSON Structure:**

```json
{
  "title": "2. Artificial Intelligence",
  "pages": "p2-p4",
  "summary": "Overview of AI paradigms and subfields.",
  "children": [
    {
      "title": "2.1 Machine Learning",
      "pages": "p3",
      "summary": "Supervised and unsupervised learning..."
    },
    {
      "title": "2.2 Deep Learning",
      "pages": "p4",
      "summary": "Neural networks and backpropagation..."
    }
  ]
}
```

---

## Pipeline 2: Query & Retrieval (Online / Reasoning)

### 1. User Query

- Takes the user's natural language question directly as input without converting it into dense vector embeddings.
- The system receives the query and starts the retrieval process.
- Its goal is to identify the most relevant sections from the JSON Tree Index.

**Example Input:**

```text
What are the differences between Machine Learning and Deep Learning?
```

### 2. LLM Tree Search (Reasoning-Based Traversal)

- The LLM inspects the top-level root nodes of the JSON Tree Index by reading their titles and summaries, rather than the entire document.
- It reasons about which branch is most relevant to the user's question.
- It drills down hierarchically from parent nodes (`2. Artificial Intelligence`) to specific child nodes (`2.1 Machine Learning`, `2.2 Deep Learning`).
- This process mimics how a human scans a book's Table of Contents to locate relevant information.

**Example Traversal Output:**

```json
{
  "reasoning": "The query asks about ML and DL, which are covered under section 2.",
  "selected_nodes": [
    {
      "title": "2.1 Machine Learning",
      "pages": "p3"
    },
    {
      "title": "2.2 Deep Learning",
      "pages": "p4"
    }
  ]
}
```

### 3. Context Extraction & Final Answer

- Once the relevant nodes are identified, the system uses their page references to locate the original document content.
- Extracts the exact raw text from the mapped pages (`p3`, `p4`).
- Passes the retrieved context to the LLM to generate a clear, synthesized answer.
- Includes **verifiable page-level citations** so users can trace the generated answer back to the original document.

---

## Overall Workflow

1. **Indexing:** Document → LLM Tree Builder → JSON Tree Index.
2. **Retrieval:** User Query → LLM Tree Search → Relevant Nodes.
3. **Answer Generation:** Page References → Context Extraction → LLM → Final Answer with Citations.

---

## Key Takeaways

- **No Vector Embeddings:** Retrieval does not depend on vector embeddings or cosine similarity.
- **Hierarchical Structure:** Preserves the document's natural organization.
- **LLM-Based Reasoning:** Uses reasoning to navigate the document tree.
- **Page-Level Retrieval:** Retrieves original content using exact page references.
- **Verifiable Citations:** Provides page references to support generated answers.